# 04 - Launch the LongGWAS pipeline

Writes a params file per dataset from the templates in `templates/`,
then prints the environment block and `nextflow run` command for each one.

Run order: the input-prep notebooks (01-03) must have finished and uploaded
their files before a dataset can be launched here; `05_meta_analysis.ipynb`
picks up once every dataset has run.

Nothing is executed from this notebook - Nextflow runs in a terminal so it
survives a kernel restart.

In [ ]:
from pathlib import Path

from gwas import config as cfg
from gwas import launch

cfg.describe()
print(f'\ntemplates:    {launch.TEMPLATE_DIR}')
print(f'params files: {launch.LAUNCH_DIR}/R{cfg.RELEASE}')

## Write the params files

`render_params` fills the placeholders in one template and writes
`launch_ready/R<release>/<label>_<analysis>.yml`.

Defaults worth knowing (see `gwas/launch.py`):

* `covar_numeric` - centred age and disease-duration terms plus PC1-PC10 for
  CS_disc, raw baseline values plus PC1-PC10 for SURV.
* `chunk_size` - 30000 variants. Lower it for a large survival cohort that
  runs out of memory.

In [ ]:
# NBA array, one ancestry at a time
for analysis in ['CS_disc', 'SURV']:
    launch.render_params(analysis, 'NBA', ancestry='EUR')

In [ ]:
# WGS-only samples for the same ancestry
for analysis in ['CS_disc', 'SURV']:
    launch.render_params(analysis, 'WGS', ancestry='EUR')

In [ ]:
# Non-NBA cohorts. geno_root is the bucket folder holding that cohort's
# normalised pgen files, written by notebook 00.
NON_NBA_COHORTS = ['DATATOP', 'OPDC', 'PARKFIT', 'PROBAND', 'SCOPA']

for cohort in NON_NBA_COHORTS:
    geno_root = cfg.gcs('non_NBA', cohort, 'PLINK')
    for analysis in ['CS_disc', 'SURV']:
        launch.render_params(analysis, 'non_NBA', cohort=cohort, geno_root=geno_root)

In [ ]:
# Small cohorts may not support ten PCs, and a large survival cohort may need
# smaller chunks. Example of overriding both:
launch.render_params(
    'SURV', 'non_NBA', cohort='OPDC',
    geno_root=cfg.gcs('non_NBA', 'OPDC', 'PLINK'),
    covar_numeric='age_at_baseline disease_duration PC1 PC2',
    chunk_size=5000,
)

## Launch one dataset

Print the environment block, paste it into the shell, then run the command
below it. `STORE_ROOT` is where the pipeline writes its results, so it has to
match the dataset being run - the meta-analysis notebook looks for results in
exactly these folders.

In [ ]:
# Pick the dataset to launch.
ANALYSIS = 'CS_disc'
MODALITY = 'NBA'        # 'NBA', 'WGS' or 'non_NBA'
ANCESTRY = 'EUR'
COHORT = None           # non-NBA only

if MODALITY == 'non_NBA':
    label, project = COHORT, f'NON_NBA_{COHORT}'
    store_root = cfg.prep_url(*cfg.prep_parts(cohort=COHORT))
elif MODALITY == 'WGS':
    label, project = f'{ANCESTRY}_WGS', f'GP2_WGS_{ANCESTRY}'
    store_root = cfg.prep_url(*cfg.prep_parts(ancestry=ANCESTRY, wgs=True))
else:
    label, project = ANCESTRY, f'GP2_{ANCESTRY}'
    store_root = cfg.prep_url(*cfg.prep_parts(ancestry=ANCESTRY))

params_file = launch.LAUNCH_DIR / f'R{cfg.RELEASE}' / f'{label}_{ANALYSIS}.yml'
print(params_file.read_text())

In [ ]:
# Credentials come from the environment - TOWER_ACCESS_TOKEN is referenced,
# never printed.
print(launch.batch_env(store_root, project))

In [ ]:
print(launch.nextflow_command(params_file, profile='gcb2'))

Once every dataset has finished, continue with `05_meta_analysis.ipynb`.